In [3]:
import numpy as np
import umap 
import matplotlib.pyplot as plt 
import torch

In [ ]:
def create_speaker_embedding(model, mel_log):

    if not isinstance(mel_log, torch.Tensor):
        mel_log = torch.tensor(mel_log)

    if mel_log.ndim == 2:
        mel_log = mel_log.unsqueeze(0) #shape: 1, time_steps
    
    with torch.no_grad():
        #extract embedding
        speaker_embeddings = model.modules.embedding_model(mel_log)
        #normalize along the embedding dimension
        speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
        #remove the middle dimension and move to CPU
        speaker_embeddings = speaker_embeddings.cpu().numpy()

    return speaker_embeddings

In [ ]:
def visualize_with_umap(embeddings, labels):
    """Generates a UMAP visualization of the embeddings

    Args:
        embeddings (np.array): an array of all speaker embeddings, shape: (N_files, embedding_size)
        labels (np.array): a 1D list with speaker IDs 
    """

    #reduce to 2D using UMAP
    umap_model = umap.UMAP(random_state=42)
    emb_tranform = umap_model.fit_transform(embeddings) #shape: (N_files, 2)

    plt.figure(figsize=(10,8))

    #get unique speakers to assign colors
    unique_speakers = np.unique(labels)

    #plot results
    for speaker in unique_speakers:
        mask = (labels == speaker) # to filter out different speakers from data
        plt.scatter(
            emb_tranform[mask, 0], # type: ignore 
            emb_tranform[mask, 1], # type: ignore 
            label = speaker, 
            s=15
            ) 

    plt.title("Speaker Embeddings mapped with UMAP", fontsize = 24)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# Load model 
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = #model path

model = resnet18()
#load saved weights into the model
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

transforms = ResNet18_Weights.DEFAULT.transforms()

print(f"\nSuccessfully loaded model from {model_path}")

In [ ]:
# Get labels
